# Capítulo 14: Árvores de Decisão

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 17 de Grus (2019).

> Uma árvore é um mistério incompreensível.
>
> — Jim Woodring

O vice-presidente de Talentos da DataSciencester entrevistou um punhado de candidatos vindos do site, com graus variados de sucesso. Ele anotou, para cada um, alguns atributos qualitativos — senioridade, linguagem preferida, se é ativo no Twitter, se tem doutorado — e se a entrevista foi bem ou mal. A pergunta que ele traz é direta: dá para usar esses dados para prever quem vai entrevistar bem, e assim parar de gastar tempo com entrevistas que não levam a nada?

Este capítulo responde com uma **árvore de decisão**, e ao fazê-lo troca de categoria de modelo. Todos os modelos ajustados até aqui foram numéricos: um vetor de parâmetros, uma função de perda, e o gradiente descendente do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) empurrando os parâmetros ladeira abaixo. A árvore não tem parâmetro contínuo, não tem função de perda diferenciável, não tem taxa de aprendizado. Ela é construída por uma **decisão gulosa repetida**: a cada passo, escolhe-se a pergunta que mais reduz a incerteza, particiona-se os dados pela resposta, e recomeça-se dentro de cada parte. O gradiente não ajuda aqui, e não é por falta — é porque não há nada contínuo para derivar.

Em compensação, a árvore ganha uma propriedade que nenhum classificador aprendido antes dela neste livro tinha: **um humano consegue lê-la**. O k-vizinhos do [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html) não explica nada — a previsão é um voto de pontos vizinhos, e a justificativa é "porque estes cinco pontos parecidos foram assim". Os coeficientes da regressão logística do [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html) dizem pouco sozinhos, e dizem menos ainda quando as variáveis estão correlacionadas. Uma árvore, desenhada numa página, **é** a explicação: você segue o caminho da raiz até a folha e vê exatamente por que aquela previsão saiu.

Essa transparência tem um preço, e ele é o assunto da última seção. Uma árvore cresce até isolar cada exemplo do treino, se você deixar — e a árvore que este capítulo constrói acerta **todos** os candidatos que usou para se construir, o que, depois do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html), você já sabe que não é boa notícia. A correção usual é a **floresta aleatória**: construir muitas árvores propositalmente diferentes e deixá-las votar. Ela funciona, e cobra de volta a legibilidade que tornava a árvore especial. A última seção mede essa troca contra a alternativa mais simples — apenas parar de crescer a árvore — e o resultado não é o que a história costuma dizer.

Ao final deste capítulo, você será capaz de:

- Explicar o que é uma árvore de decisão e por que ela não se ajusta por gradiente descendente
- Calcular a entropia de um conjunto rotulado e interpretá-la como incerteza medida em bits
- Calcular a entropia de uma partição e usá-la para escolher o melhor atributo de divisão
- Reconhecer por que um atributo com muitos valores distintos engana esse critério
- Implementar o algoritmo ID3 e explicar em que sentido ele é guloso, e o que isso custa
- Prever com uma árvore construída e identificar o que ela faz diante de um valor que nunca viu
- Construir uma floresta aleatória a partir de duas fontes de aleatoriedade, e medir o que ela ganha — e o que ela não ganha — contra uma árvore podada

## Seções

| Seção | Tópico |
|---|---|
| [14.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/01-o-que-e-uma-arvore-de-decisao.html) | O que é uma Árvore de Decisão? |
| [14.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/02-entropia.html) | Entropia |
| [14.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/03-a-entropia-de-uma-particao.html) | A Entropia de uma Partição |
| [14.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/04-criando-uma-arvore.html) | Criando uma Árvore de Decisão |
| [14.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/05-juntando-tudo.html) | Juntando Tudo |
| [14.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html) | Florestas Aleatórias |

## O que é uma Árvore de Decisão?

> **📌 Nota**
>
> Esta seção corresponde a *What Is a Decision Tree?*, do capítulo 17 de Grus (2019).

Uma árvore de decisão usa uma estrutura de árvore para representar vários **caminhos de decisão** possíveis, e um resultado para cada caminho.

Se você já jogou vinte perguntas, já usou uma. O diálogo é mais ou menos assim:

- "Estou pensando num animal."
- "Tem mais de cinco patas?"
- "Não."
- "É gostoso?"
- "Não."
- "Sai na moeda australiana de cinco centavos?"
- "Sim."
- "É um equidna?"
- "Sim, é!"

O caminho percorrido foi:

> "Não mais que 5 patas" → "Não é gostoso" → "Sai na moeda de 5 centavos" → "Equidna!"

Esse caminho existe dentro de uma árvore de "adivinhe o animal" idiossincrática e nada abrangente:

In [ ]:
# Figura: Uma árvore de decisão para adivinhar o animal. Cada caixa clara é uma pergunta; cada caixa azul, uma resposta final.
from matplotlib import pyplot as plt

# Uma árvore aqui é (pergunta, {resposta: subárvore}); uma folha é uma string.
bicho = ("Mais de 5 patas?", {
    "não": ("É gostoso?", {
        "não": ("Sai na moeda australiana de 5 centavos?",
                {"não": "Gatinho!", "sim": "Equidna!"}),
        "sim": ('Estrela de "A Teia de Charlotte"?',
                {"não": "Bisão!", "sim": "Porco!"}),
    }),
    "sim": ("Vive embaixo da sua cama?", {
        "não": ("Faz mel?", {"não": "Mosquito!", "sim": "Abelha!"}),
        "sim": ('Estrela de "A Teia de Charlotte"?',
                {"não": "Percevejo!", "sim": "Aranha!"}),
    }),
})

def folhas(t):
    return 1 if isinstance(t, str) else sum(folhas(s) for s in t[1].values())

def altura(t):
    return 0 if isinstance(t, str) else 1 + max(altura(s) for s in t[1].values())

def desenha(ax, t, x0, x1, prof, h):
    """Posiciona cada nó: y pela profundidade, x pela fatia de folhas abaixo dele."""
    y, x = h - prof, (x0 + x1) / 2
    if isinstance(t, str):
        ax.text(x, y, t, ha='center', va='center', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.35', fc='#dbe9f6', ec='#4a7fb5'))
        return x, y
    ax.text(x, y, t[0], ha='center', va='center', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.35', fc='#f6ead0', ec='#b58a4a'))
    total, ini = folhas(t), x0
    for resposta, sub in t[1].items():
        larg = (x1 - x0) * folhas(sub) / total
        xs, ys = desenha(ax, sub, ini, ini + larg, prof + 1, h)
        ax.annotate("", xy=(xs, ys + 0.18), xytext=(x, y - 0.18),
                    arrowprops=dict(arrowstyle='->', color='#888', lw=1))
        ax.text((x + xs) / 2, (y + ys) / 2, resposta, fontsize=7, color='#444',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.9))
        ini += larg
    return x, y

h = altura(bicho)
fig, ax = plt.subplots(figsize=(11, 1.15 * h + 0.8))
desenha(ax, bicho, 0, 1, 0, h)
ax.set_xlim(-0.06, 1.06)
ax.set_ylim(-0.35, h + 0.35)
ax.axis('off')
plt.tight_layout()
plt.show()

Cada caixa clara é um **nó de decisão**: ela faz uma pergunta e nos manda por um caminho diferente conforme a resposta. Cada caixa azul é uma **folha**: ela não pergunta nada, apenas devolve uma previsão.

### O que as árvores têm de bom

Árvores de decisão têm muito a seu favor.

> **🔷 Conceito**
>
> São muito fáceis de entender e interpretar, e o processo pelo qual chegam a uma previsão é **completamente transparente**. Ao contrário de todos os outros modelos vistos até aqui, uma árvore lida naturalmente com uma mistura de atributos numéricos (número de patas) e categóricos (gostoso ou não), e consegue até classificar dados em que alguns atributos estão faltando.

Essa lista descreve **árvores de decisão em geral**, e vale separá-la do que este capítulo vai construir: o ID3 entrega o primeiro item por inteiro e nenhum dos outros dois. Ele só sabe fazer uma pergunta — "qual é o valor deste atributo?" —, o que trata uma coluna numérica como se fosse categórica, e não tem tratamento nenhum para um valor faltante. As duas lacunas têm conserto conhecido, e é ele que o `scikit-learn` implementa; a [seção 14.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/04-criando-uma-arvore.html) mostra qual é.

Esse primeiro ponto merece ser dito com todas as letras, porque é a novidade do capítulo. Você já construiu quatro classificadores neste livro, e nenhum deles entregava uma explicação em que se pudesse confiar:

- O classificador improvisado da [seção 1.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/03-hipotese-motivadora-datasciencester.html) — `if years_experience < 3.0 ... elif years_experience < 8.5 ...` — parece a exceção, porque se lê numa olhada. Mas ninguém o **ajustou**: os cortes foram escolhidos à mão, lidos do próprio conjunto de dados que ele deveria explicar, olhando onde as respostas mudavam. O que se lê ali não é conhecimento sobre o mundo — é o conjunto de treino decorado, o defeito que o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) batizou. Legibilidade sem aprendizado não é o que este capítulo promete.
- O k-vizinhos da [seção 9.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-o-modelo.html) responde "porque os cinco pontos mais parecidos com este foram assim". É uma justificativa, mas não é uma explicação: ela não diz *o que*, nos dados, importou.
- O Naive Bayes do [Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html) devolve um produto de milhares de probabilidades condicionais. Dá para inspecionar as palavras mais indicativas, mas a decisão em si é uma conta que ninguém refaz de cabeça.
- A regressão logística do [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html) devolve um vetor de coeficientes. Eles têm interpretação — mas só sob hipóteses sobre a escala e a independência das variáveis, e o [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) mostrou como essa interpretação desmorona quando duas variáveis andam juntas.

Uma árvore não precisa de nada disso. A figura acima **é** o modelo. Não há uma representação separada, mais fiel, escondida atrás dela.

### O que as árvores têm de ruim

Duas coisas, e as duas moldam o resto do capítulo.

A primeira: encontrar a árvore **ótima** para um conjunto de dados é um problema computacionalmente muito difícil. Vamos contornar isso construindo uma árvore boa o bastante em vez de uma ótima — que já dá bastante trabalho para conjuntos grandes.

A segunda, mais importante: é muito fácil (e muito ruim) construir árvores **sobreajustadas** aos dados de treino, que não generalizam para dados novos. Essa é a mesma armadilha que o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) descreveu em abstrato, e a árvore é o modelo deste livro que cai nela com mais facilidade — ela tem, literalmente, a capacidade de fazer uma pergunta por exemplo do treino até que cada um fique sozinho na sua folha. A [seção 14.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html) trata disso.

### O que vamos construir

A maioria das pessoas divide as árvores de decisão em **árvores de classificação**, que produzem saídas categóricas, e **árvores de regressão**, que produzem saídas numéricas. Aqui vamos nos concentrar nas de classificação, e percorrer o algoritmo **ID3** para aprender uma árvore a partir de dados rotulados. Para simplificar, os problemas serão de saída binária: "devo contratar este candidato?", "devo mostrar a propaganda A ou a B para este visitante?", "comer esta comida que achei na geladeira do escritório vai me deixar doente?".

> **❗ Importante — Este capítulo não usa gradiente descendente, e isso não é um detalhe**
>
> Do [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html) ao [13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html), o roteiro foi sempre o mesmo: escrever um modelo com parâmetros, escrever uma função de perda, derivar o gradiente, descer. A árvore quebra esse roteiro por completo, e o motivo é estrutural, não uma escolha de implementação: **não existe parâmetro contínuo para derivar**. A "escolha" que o algoritmo faz a cada passo é qual atributo usar para dividir os dados — uma escolha entre um número finito de alternativas discretas, sobre a qual não se pode calcular uma derivada.
>
> No lugar do gradiente entra uma **busca gulosa**: em cada nó, avalia-se cada atributo disponível, escolhe-se o melhor pelo critério da [seção 14.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/03-a-entropia-de-uma-particao.html), e nunca se revê a escolha. É por isso que o [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) já avisava, na abertura, que as árvores de decisão estavam entre os modelos que não passariam por ele.

> **💡 Dica — Na prática: o que se faz com isso**
>
> Uma árvore de decisão sozinha raramente é o modelo final de um sistema em produção — ela é frágil demais, pelo motivo que a [seção 14.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html) vai medir. O que domina hoje o aprendizado sobre dados tabulares são os **comitês de árvores**: muitas árvores votando, em vez de uma decidindo. Em dados em formato de tabela — a maior parte do que existe numa empresa —, esses métodos costumam bater redes neurais, e por margem confortável. A [seção 14.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html) constrói o mais simples deles.
>
> A árvore isolada continua tendo dois usos reais. O primeiro é ser **auditável**: em contextos onde a decisão precisa ser justificada a um humano — crédito, admissão, triagem —, uma árvore rasa que se lê numa página vale mais do que um modelo mais preciso que não se explica. O segundo é ser um **diagnóstico**: treinar uma árvore rasa nos seus dados e olhar os primeiros dois níveis é uma forma rápida e barata de descobrir quais atributos carregam sinal, antes de escolher o modelo de verdade.
>
> Um aviso para quando você for comparar com a biblioteca: a árvore do `scikit-learn` **não** é a deste capítulo. O `DecisionTreeClassifier` implementa uma variante do CART, e o ID3 que vamos escrever é outro algoritmo. As diferenças que importam ficam no callout de fechamento da [seção 14.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/05-juntando-tudo.html), depois de você ter a nossa árvore construída para comparar com a dele — que é a hora em que elas dizem alguma coisa.

## Entropia

> **📌 Nota**
>
> Esta seção corresponde a *Entropy*, do capítulo 17 de Grus (2019).

Para construir uma árvore de decisão, precisamos decidir **que perguntas fazer** e **em que ordem**. Em cada estágio da árvore há possibilidades que já eliminamos e possibilidades que ainda não. Depois de descobrir que um animal não tem mais de cinco patas, eliminamos a possibilidade de ser um gafanhoto; não eliminamos a possibilidade de ser um pato. Cada pergunta possível particiona as possibilidades restantes de acordo com a resposta.

O que queremos, idealmente, é escolher perguntas cujas respostas deem muita informação sobre o que o modelo deve prever. Se existisse uma única pergunta de sim/não cujas respostas "sim" correspondessem sempre a saídas `True` e cujas respostas "não" correspondessem sempre a saídas `False`, ela seria uma pergunta ótima: uma pergunta e acabou. No outro extremo, uma pergunta cujas duas respostas não digam nada de novo sobre a previsão é uma pergunta desperdiçada.

Essa noção de "quanta informação" tem um nome: **entropia**.

### Incerteza, medida em bits

Você provavelmente já ouviu a palavra entropia usada como sinônimo de desordem. Aqui ela representa algo mais preciso: a **incerteza associada aos dados**.

Imagine um conjunto $S$ de dados, cada elemento rotulado como pertencente a uma de $n$ classes $c_1, \dots, c_n$. Se todos os pontos pertencem a uma única classe, não há incerteza real, e queremos que a entropia seja baixa. Se os pontos estão espalhados uniformemente pelas classes, há muita incerteza, e queremos que a entropia seja alta.

> **🔷 Conceito**
>
> Se $p_i$ é a proporção de dados rotulados como classe $c_i$, a **entropia** de $S$ é
>
> $$H(S) = -p_1 \log_2 p_1 - \dots - p_n \log_2 p_n$$
>
> com a convenção usual de que $0 \log 0 = 0$.
>
> O logaritmo na base 2 dá à conta uma **unidade**, e a unidade é o **bit** — que aqui tem leitura direta: $H(S)$ é o número médio de perguntas de sim/não que ainda faltam para descobrir a classe de um elemento sorteado de $S$, perguntando do jeito mais esperto possível. Zero bit é certeza e nenhuma pergunta; um bit é exatamente uma pergunta.
>
> Repare no que a fórmula **não** olha: ela não usa os rótulos, só as proporções. Um conjunto de 3 spams e 1 não-spam tem a mesma entropia de um conjunto de 3 gatos e 1 cachorro. O que ela mede é o quanto os dados estão espalhados entre as classes, e nada mais.

Cada termo $-p_i \log_2 p_i$ é não negativo e fica perto de zero exatamente quando $p_i$ está perto de 0 ou perto de 1. Vale a pena olhar as duas coisas ao mesmo tempo — o comportamento de um termo isolado e o da soma, no caso de duas classes:

In [ ]:
# Figura: À esquerda, a contribuição de uma única classe. À direita, a entropia total de um conjunto com duas classes de proporções $p$ e $1-p$.
import math
from matplotlib import pyplot as plt

def entropy(class_probabilities):
    return sum(-p * math.log(p, 2) for p in class_probabilities if p > 0)

ps = [i / 1000 for i in range(1, 1000)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))

ax1.plot(ps, [-p * math.log(p, 2) for p in ps], color='tab:blue')
ax1.set_xlabel("$p$")
ax1.set_ylabel(r"$-p \log_2 p$")
ax1.set_title("A contribuição de uma classe", fontsize=10)
ax1.axhline(0, color='#bbb', lw=0.8)

ax2.plot(ps, [entropy([p, 1 - p]) for p in ps], color='tab:red')
ax2.set_xlabel("$p$")
ax2.set_ylabel("$H$ (bits)")
ax2.set_title("Entropia de duas classes: $p$ e $1-p$", fontsize=10)
ax2.axvline(0.5, color='#bbb', ls='--', lw=1)
ax2.axhline(1.0, color='#bbb', ls='--', lw=1)
ax2.plot([0.25], [entropy([0.25, 0.75])], 'ko', ms=4)
ax2.annotate("0,811", xy=(0.25, entropy([0.25, 0.75])), xytext=(0.06, 0.62),
             fontsize=8, arrowprops=dict(arrowstyle='->', color='#666'))

plt.tight_layout()
plt.show()

O painel da esquerda mostra que um termo isolado vale zero nos dois extremos e é máximo perto de $p \approx 0{,}37$. O da direita mostra a consequência para o conjunto todo: a entropia é **zero** quando uma das classes leva tudo, cresce à medida que as duas se equilibram, e atinge o **máximo de 1 bit** exatamente na divisão meio a meio. É esse o comportamento que queremos: entropia pequena quando quase tudo está numa única classe, entropia grande quando os dados estão espalhados.

### A função

Enrolar tudo isso numa função é fácil:

In [ ]:
from typing import List
import math

def entropy(class_probabilities: List[float]) -> float:
    """Dada uma lista de probabilidades de classe, calcula a entropia"""
    return sum(-p * math.log(p, 2)
               for p in class_probabilities
               if p > 0)                     # ignora probabilidades zero

assert entropy([1.0]) == 0
assert entropy([0.5, 0.5]) == 1
assert 0.81 < entropy([0.25, 0.75]) < 0.82

O `if p > 0` é a convenção $0 \log 0 = 0$ escrita em código: `math.log(0, 2)` estouraria, e o termo correspondente vale zero de qualquer forma.

Os três valores das asserções contam a história inteira:

In [ ]:
for ps in [[1.0], [0.5, 0.5], [0.25, 0.75]]:
    print(f"entropy({ps}) = {entropy(ps):.6f}")

- `entropy([1.0])` é **exatamente 0**. Uma única classe, certeza total, nenhuma surpresa possível. Você não precisa perguntar nada: já sabe a resposta.
- `entropy([0.5, 0.5])` é **exatamente 1**. Duas classes igualmente prováveis, o pior caso possível para duas classes. Você precisa de uma pergunta de sim/não para descobrir qual é.
- `entropy([0.25, 0.75])` dá **0,811278**. Desequilibrado, mas não decidido: há incerteza, só que menos do que no caso meio a meio.

> **🟩 Exemplo — Por que o logaritmo é na base 2**
>
> A escolha da base não muda qual conjunto tem mais entropia que qual — trocar a base multiplica todos os valores pela mesma constante. O que a base 2 dá é a **unidade interpretável** anunciada no conceito acima: o bit, a resposta de uma pergunta de sim/não.
>
> Vale conferir que a contagem de perguntas fecha:

In [ ]:
print(f"2 classes iguais: {entropy([0.5, 0.5]):.6f} bits")
print(f"4 classes iguais: {entropy([0.25] * 4):.6f} bits")
print(f"3 classes iguais: {entropy([1/3] * 3):.6f} bits")

> Com quatro classes igualmente prováveis, dois bits: "está na primeira metade?", depois "está na primeira metade do que sobrou?". Com três, o valor é 1,585 — um número quebrado porque não existe uma sequência de perguntas de sim/não que resolva três casos igualmente prováveis em número inteiro de perguntas; 1,585 é a média que você atinge no limite, perguntando sobre muitos sorteios de uma vez.
>
> Isso não é curiosidade de rodapé: é exatamente o que estamos otimizando. A [seção 14.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/04-criando-uma-arvore.html) vai construir a árvore escolhendo, a cada passo, a pergunta que mais reduz esse número — ou seja, **o algoritmo joga vinte perguntas da forma mais eficiente que consegue**.

### Da lista de probabilidades para os dados

A função `entropy` recebe probabilidades, mas nossos dados vão ser pares `(entrada, rótulo)`. Precisamos calcular as proporções nós mesmos — e, de novo, sem olhar para quais rótulos são, apenas para as proporções:

In [ ]:
from typing import Any
from collections import Counter

def class_probabilities(labels: List[Any]) -> List[float]:
    total_count = len(labels)
    return [count / total_count
            for count in Counter(labels).values()]

def data_entropy(labels: List[Any]) -> float:
    return entropy(class_probabilities(labels))

assert data_entropy(['a']) == 0
assert data_entropy([True, False]) == 1
assert data_entropy([3, 4, 4, 4]) == entropy([0.25, 0.75])

A última asserção é a que vale reler: uma lista com um `3` e três `4` tem exatamente a mesma entropia que a distribuição `[0.25, 0.75]`. Os valores `3` e `4` não entram na conta em lugar nenhum — só a contagem entra.

> **💡 Dica — Na prática: entropia, e a alternativa que a biblioteca usa por padrão**
>
> A entropia já está pronta em `scipy`, e com a base como parâmetro:
>
> ```python
> from scipy.stats import entropy
>
> entropy([0.25, 0.75], base=2)   # 0.8112781244591328
> entropy([1, 3], base=2)         # o mesmo: normaliza as contagens sozinha
> ```
>
> O detalhe mais útil aqui não é a função, é a **alternativa**. O `scikit-learn` deixa você escolher o critério de divisão da árvore:
>
> ```python
> from sklearn.tree import DecisionTreeClassifier
>
> DecisionTreeClassifier(criterion='entropy')   # o deste capítulo
> DecisionTreeClassifier(criterion='gini')      # o padrão
> ```
>
> O padrão **não** é a entropia: é a **impureza de Gini**, $1 - \sum_i p_i^2$, que mede a mesma coisa com outra fórmula. Ela também vale 0 quando uma classe leva tudo e é máxima na divisão uniforme; a diferença é que não tem logaritmo, e por isso é mais barata de calcular — o que importa quando ela é avaliada milhões de vezes durante a construção de uma floresta.
>
> Na prática, as duas escolhem quase sempre a mesma divisão, e a literatura empírica não mostra vantagem consistente de nenhuma das duas. A entropia é a que este capítulo usa porque é a que se **explica**: ela vem da teoria da informação, tem unidade (bits), e a sua unidade responde à pergunta "quantas perguntas de sim/não faltam?". O Gini não tem essa leitura — é uma medida de impureza que funciona, sem uma história por trás.

## A Entropia de uma Partição

> **📌 Nota**
>
> Esta seção corresponde a *The Entropy of a Partition*, do capítulo 17 de Grus (2019).

O que a [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/02-entropia.html) fez foi calcular a entropia — a incerteza — de **um único** conjunto de dados rotulados. Só que cada estágio de uma árvore de decisão consiste em fazer uma pergunta cuja resposta **particiona** os dados em dois ou mais subconjuntos. A pergunta "tem mais de cinco patas?" separa os animais entre os que têm (aranhas) e os que não têm (equidnas).

Precisamos, então, de uma noção de entropia **da partição**, e não de um conjunto só.

### A armadilha da média simples

O que queremos de uma boa partição é que ela divida os dados em subconjuntos que, eles próprios, tenham entropia baixa — ou seja, que sejam bem decididos. Uma partição ruim é a que produz subconjuntos grandes e incertos.

O detalhe está no "grandes". A pergunta "sai na moeda australiana de cinco centavos?" era, na verdade, bem burra (embora sortuda): ela particionava os animais restantes em $S_1 = \{\text{equidna}\}$ e $S_2 = \{\text{todo o resto}\}$. O subconjunto $S_1$ tem entropia zero — perfeito! —, mas $S_2$ é ao mesmo tempo **grande** e de entropia alta. Se calculássemos a média simples das duas entropias, essa pergunta pareceria excelente, porque um dos lados é perfeito e a média puxa para baixo.

> **🔷 Conceito**
>
> Matematicamente, se particionamos os dados $S$ em subconjuntos $S_1, \dots, S_m$ contendo proporções $q_1, \dots, q_m$ dos dados, a entropia da partição é a **soma ponderada**
>
> $$H = q_1 H(S_1) + \dots + q_m H(S_m)$$
>
> O peso $q_i$ é o que impede a armadilha: um subconjunto de entropia zero com um único elemento entra na conta com peso $1/n$, e quase não desconta nada. Só reduz a entropia da partição quem deixa uma **fatia grande** dos dados decidida.

Em código:

In [ ]:
from typing import Any, List
import math
from collections import Counter

def entropy(class_probabilities: List[float]) -> float:
    """Dada uma lista de probabilidades de classe, calcula a entropia"""
    return sum(-p * math.log(p, 2) for p in class_probabilities if p > 0)

def class_probabilities(labels: List[Any]) -> List[float]:
    total_count = len(labels)
    return [count / total_count for count in Counter(labels).values()]

def data_entropy(labels: List[Any]) -> float:
    return entropy(class_probabilities(labels))

def partition_entropy(subsets: List[List[Any]]) -> float:
    """Devolve a entropia desta partição dos dados em subconjuntos"""
    total_count = sum(len(subset) for subset in subsets)

    return sum(data_entropy(subset) * len(subset) / total_count
               for subset in subsets)

As três primeiras funções são as da [seção 14.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/02-entropia.html), sem alteração nenhuma. A novidade é só a última: ela pesa a entropia de cada pedaço pelo tamanho do pedaço.

### Os dados do vice-presidente de Talentos

Chegou a hora de olhar dados de verdade. O vice-presidente entregou os registros das entrevistas: para cada candidato, a senioridade (`level`), a linguagem preferida (`lang`), se é ativo no Twitter (`tweets`), se tem doutorado (`phd`) e se a entrevista foi bem (`did_well`).

In [ ]:
from typing import NamedTuple, Optional

class Candidate(NamedTuple):
    level: str
    lang: str
    tweets: bool
    phd: bool
    did_well: Optional[bool] = None  # permite dados não rotulados

                  #  level     lang     tweets  phd  did_well
inputs = [Candidate('Senior', 'Java',   False, False, False),
          Candidate('Senior', 'Java',   False, True,  False),
          Candidate('Mid',    'Python', False, False, True),
          Candidate('Junior', 'Python', False, False, True),
          Candidate('Junior', 'R',      True,  False, True),
          Candidate('Junior', 'R',      True,  True,  False),
          Candidate('Mid',    'R',      True,  True,  True),
          Candidate('Senior', 'Python', False, False, False),
          Candidate('Senior', 'R',      True,  False, True),
          Candidate('Junior', 'Python', True,  False, True),
          Candidate('Senior', 'Python', True,  True,  True),
          Candidate('Mid',    'Python', False, True,  True),
          Candidate('Mid',    'Java',   True,  False, True),
          Candidate('Junior', 'Python', False, True,  False)]

len(inputs)

São 14 candidatos, e o `did_well` está distribuído assim:

In [ ]:
Counter(candidato.did_well for candidato in inputs)

Nove entrevistas boas e cinco ruins. Antes de perguntar qualquer coisa, a incerteza sobre um candidato qualquer é:

In [ ]:
data_entropy([candidato.did_well for candidato in inputs])

Cerca de **0,940 bit**. É o ponto de partida: qualquer pergunta útil precisa deixar esse número menor.

### Particionando por um atributo

Dois utilitários bastam. O primeiro agrupa as entradas pelo valor de um atributo; o segundo calcula a entropia da partição resultante.

In [ ]:
from typing import Dict, TypeVar
from collections import defaultdict

T = TypeVar('T')  # tipo genérico para as entradas

def partition_by(inputs: List[T], attribute: str) -> Dict[Any, List[T]]:
    """Particiona as entradas em listas conforme o atributo especificado."""
    partitions: Dict[Any, List[T]] = defaultdict(list)
    for input in inputs:
        key = getattr(input, attribute)  # valor do atributo especificado
        partitions[key].append(input)    # põe a entrada na partição certa
    return partitions

def partition_entropy_by(inputs: List[Any],
                         attribute: str,
                         label_attribute: str) -> float:
    """Calcula a entropia correspondente à partição dada"""
    # as partições são feitas das nossas entradas
    partitions = partition_by(inputs, attribute)

    # mas partition_entropy precisa só dos rótulos de classe
    labels = [[getattr(input, label_attribute) for input in partition]
              for partition in partitions.values()]

    return partition_entropy(labels)

Agora dá para perguntar, a cada um dos quatro atributos, quanta incerteza ele elimina:

In [ ]:
for key in ['level', 'lang', 'tweets', 'phd']:
    print(f"{key:8s} {partition_entropy_by(inputs, key, 'did_well'):.6f}")

| Atributo | Entropia da partição |
|---|---|
| `level` | **0,693536** |
| `tweets` | 0,788450 |
| `lang` | 0,860132 |
| `phd` | 0,892159 |

Os quatro ficam abaixo da incerteza inicial de 0,940 bit, e é tentador ler isso como "todos carregam alguma informação". **Não carrega essa conclusão**: a entropia de uma partição nunca é maior que a do conjunto inteiro — o ganho é não negativo por construção —, então "reduziu a incerteza" não distingue sinal de ruído. Nenhum atributo poderia ter falhado nesse teste.

Dá para medir o quanto isso engana. Acrescentando aos mesmos 14 candidatos uma coluna sorteada ao acaso entre três valores, sem relação nenhuma com o alvo, o ganho médio em 1.000 sorteios é de **0,131 bit** — e nunca deu zero, nem uma vez; o menor sorteio ficou em 0,0013. Ou seja: nesta base, ruído puro rende mais que o `phd`, cujo ganho é de 0,048 bit. Com 14 exemplos, quase tudo parece informativo.

O que a tabela sustenta é a comparação, não o julgamento individual: o menor de todos, e portanto o vencedor, é `level`. Particionar os candidatos por senioridade deixa a incerteza média em **0,694 bit**, uma redução de cerca de 0,247 bit.

A partição mostra por quê:

In [ ]:
for valor, grupo in partition_by(inputs, 'level').items():
    rotulos = [c.did_well for c in grupo]
    print(f"{valor:7s} n={len(grupo)}  {dict(Counter(rotulos))}  "
          f"H={data_entropy(rotulos):.4f}")

O grupo `Mid` é o herói silencioso: os quatro candidatos de nível intermediário se saíram bem, **todos**, então esse pedaço da partição tem entropia zero e carrega 4 dos 14 candidatos consigo. Os outros dois grupos continuam quase indecisos (3 contra 2, em ambos), mas juntos já não pesam o suficiente para estragar a média.

### O que acontece dentro de um ramo

Escolhido `level` como primeira pergunta, cada grupo vira um problema novo — menor e com um atributo a menos disponível. Olhemos os cinco candidatos sêniores:

In [ ]:
senior_inputs = [entrada for entrada in inputs if entrada.level == 'Senior']

print(f"entropia dos {len(senior_inputs)} sêniores: "
      f"{data_entropy([c.did_well for c in senior_inputs]):.6f}\n")

for key in ['lang', 'tweets', 'phd']:
    print(f"{key:8s} {partition_entropy_by(senior_inputs, key, 'did_well'):.6f}")

In [ ]:
# Figura: Entropia da partição por atributo. A linha tracejada é a entropia antes de perguntar qualquer coisa; o vencedor está em verde — nome, barra e valor. No painel da direita a barra vencedora tem largura zero, que é justamente o ponto: `tweets` deixa os sêniores sem nenhuma incerteza.
from matplotlib import pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4))

def painel(ax, dados, atributos, titulo):
    hs = [partition_entropy_by(dados, a, 'did_well') for a in atributos]
    antes = data_entropy([d.did_well for d in dados])
    cores = ['tab:green' if h == min(hs) else '#9ab' for h in hs]
    barras = ax.barh(atributos, hs, color=cores)
    ax.axvline(antes, color='tab:red', ls='--', lw=1.2)
    # o rótulo da linha fica à ESQUERDA dela, encostado: ancorado no canto
    # direito do eixo ele se estendia para cá e saía cortado pela própria linha
    ax.text(antes / 1.1 - 0.01, 0.94, f"antes: {antes:.3f}".replace('.', ','),
            color='tab:red', fontsize=8, ha='right', va='top',
            transform=ax.transAxes)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("entropia da partição (bits)")
    ax.set_title(titulo, fontsize=10)
    # o vencedor pode ter entropia zero, e aí a barra some: o nome e o valor
    # em verde e em negrito são o que o torna visível na figura.
    for rotulo, h in zip(ax.get_yticklabels(), hs):
        if h == min(hs):
            rotulo.set_color('tab:green')
            rotulo.set_fontweight('bold')
    # o valor vai DENTRO da barra quando ela é larga o bastante para caber:
    # posto do lado de fora, ele era atravessado pela linha vermelha sempre que
    # a entropia da partição chegava perto da entropia de antes
    for b, h in zip(barras, hs):
        venceu = h == min(hs)
        dentro = h > 0.25
        ax.text(h - 0.02 if dentro else h + 0.02,
                b.get_y() + b.get_height() / 2,
                f"{h:.3f}".replace('.', ','),
                va='center', ha='right' if dentro else 'left', fontsize=8,
                # dentro da barra o texto é branco -- inclusive na vencedora,
                # que é verde: verde sobre verde some
                color='white' if dentro else ('tab:green' if venceu else 'black'),
                fontweight='bold' if venceu else 'normal')

painel(ax1, inputs, ['level', 'lang', 'tweets', 'phd'], "Os 14 candidatos")
painel(ax2, senior_inputs, ['lang', 'tweets', 'phd'], "Só os 5 sêniores")

plt.tight_layout()
plt.show()

Os cinco sêniores começam com entropia 0,971 — quase o máximo de 1 bit, já que são 3 contra 2. E aí acontece o melhor momento do capítulo:

> **🔷 Conceito**
>
> Particionar os sêniores por `tweets` dá entropia **exatamente 0,000000**.
>
> Zero não é "muito bom". Zero é o valor que a entropia só assume quando **cada lado da partição contém uma única classe** — quando não sobrou nenhuma dúvida em nenhum dos pedaços. Como a entropia da partição é uma soma ponderada de entropias não negativas, ela só dá zero se cada parcela der zero, e cada parcela só dá zero se aquele subconjunto for puro.
>
> Em outras palavras: entre os sêniores, quem tuita se saiu bem, e quem não tuita se saiu mal. Sem exceção. **Não há mais nada a perguntar** — o ramo acabou ali.

Vale conferir a afirmação diretamente, em vez de acreditar no número:

In [ ]:
for valor, grupo in partition_by(senior_inputs, 'tweets').items():
    print(f"tweets={str(valor):5s} n={len(grupo)}  "
          f"{dict(Counter(c.did_well for c in grupo))}")

Três sêniores que não tuitam, os três com entrevista ruim; dois que tuitam, os dois com entrevista boa. É essa pureza que o `0.000000` estava reportando.

> **⚠️ Atenção**
>
> Cinco pessoas. Antes de sair contando a alguém que tuitar prevê o desempenho de sêniores em entrevista, releia a frase anterior: **cinco pessoas**. Uma regra perfeita sobre cinco exemplos é o resultado mais fácil de obter e o menos confiável que existe — com quatro atributos e cinco pontos, encontrar uma separação perfeita por acaso é quase esperado.
>
> A árvore vai adotar essa regra assim mesmo, porque é isso que o algoritmo faz: ele otimiza sobre os dados que recebeu, e não tem como saber que recebeu poucos. Esse é precisamente o mecanismo do sobreajuste do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html), visto de perto, e é o problema que a [seção 14.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html) vai atacar.

### O atributo que engana o critério

Há um problema sério com esta abordagem, e ele merece um exemplo executado.

Particionar por um atributo com **muitos valores distintos** produz entropia baixíssima quase de graça — no limite, subconjuntos de uma pessoa cada, todos necessariamente de entropia zero. Imagine que você trabalha num banco e quer prever inadimplência a partir de dados históricos. Se o conjunto contém o CPF de cada cliente, particionar por CPF produz subconjuntos de uma pessoa, entropia zero, vitória esmagadora sobre qualquer outro atributo.

Vamos fazer exatamente isso com os candidatos, acrescentando um número de matrícula sem significado nenhum:

In [ ]:
class CandidatoComId(NamedTuple):
    id: int
    level: str
    lang: str
    tweets: bool
    phd: bool
    did_well: Optional[bool] = None

com_id = [CandidatoComId(n, c.level, c.lang, c.tweets, c.phd, c.did_well)
          for n, c in enumerate(inputs)]

for key in ['id', 'level', 'lang', 'tweets', 'phd']:
    print(f"{key:8s} {partition_entropy_by(com_id, key, 'did_well'):.6f}")

O `id` vence por goleada: **0,000000** contra os 0,694 do `level`. Pelo critério que acabamos de escrever, o número de matrícula é o melhor preditor de desempenho em entrevista de que dispomos.

> **⚠️ Atenção**
>
> Um modelo que depende do `id` **tem certeza** de não generalizar. Ele acerta 100% dos dados de treino e não tem nada a dizer sobre o candidato número 15, porque nunca viu o número 15.
>
> O critério de entropia não erra a conta — ele calcula exatamente o que promete. O que ele não sabe é a diferença entre "este atributo explica o fenômeno" e "este atributo identifica a linha". Nada na fórmula distingue as duas coisas, e é por isso que a responsabilidade fica com você: **evite (ou agrupe em faixas) atributos com muitos valores distintos ao construir árvores de decisão**.
>
> Isso não é uma peculiaridade de um dataset de brinquedo. É a mesma classe de erro do classificador da [seção 1.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/03-hipotese-motivadora-datasciencester.html), que lia dos próprios dados que deveria explicar os cortes de anos de experiência — e a mesma coisa que um identificador de usuário, um carimbo de tempo com precisão de milissegundo ou um número de pedido fazem, discretamente, com qualquer modelo em que entrem por acidente.
>
> Decidir o que entra como atributo é, portanto, uma etapa do trabalho e não um detalhe de preparação de dados. É o assunto da [seção 8.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-extracao-e-selecao-de-atributos.html), e este é o exemplo mais nítido dela em todo o livro: a coluna com o maior ganho de informação é a que precisa ser removida.

> **💡 Dica — Na prática: `mutual_info_classif`**
>
> A quantidade que estamos calculando tem nome próprio na teoria da informação. A diferença entre a entropia antes de perguntar e a entropia da partição depois de perguntar chama-se **ganho de informação**, e é o mesmo que a **informação mútua** entre o atributo e o alvo. Para `level`, ela vale $0{,}940286 - 0{,}693536 = 0{,}246750$ bit.
>
> O `scikit-learn` a oferece pronta, no módulo de seleção de atributos:
>
> ```python
> from sklearn.feature_selection import mutual_info_classif
>
> # level codificado como 0/1/2, alvo como 0/1
> mutual_info_classif(X, y, discrete_features=True, random_state=0)
> ```
>
> Duas diferenças importam ao comparar com o nosso número. A primeira é a **unidade**: o `scikit-learn` devolve *nats* (logaritmo natural), não bits — para `level`, ele devolve `0.17103394`, que dividido por $\ln 2$ dá exatamente os nossos `0.24674982`. A segunda é que, sem `discrete_features=True`, ele usa um estimador baseado em vizinhos mais próximos, apropriado para atributos contínuos, e o resultado deixa de bater com a conta discreta.
>
> Repare no que isso significa para o pipeline usual: rodar `mutual_info_classif` sobre as colunas e ficar com as melhores é fazer, uma vez só e por atributo isolado, exatamente o que a árvore faz recursivamente em cada nó. E herda o mesmo defeito — uma coluna de identificadores lidera o ranking com folga.

## Criando uma Árvore de Decisão

> **📌 Nota**
>
> Esta seção corresponde a *Creating a Decision Tree*, do capítulo 17 de Grus (2019).

Temos os dados e temos um critério para julgar uma pergunta. Falta o algoritmo que junta os dois e produz uma árvore.

Nossa árvore vai consistir de **nós de decisão** (que fazem uma pergunta e nos mandam por caminhos diferentes conforme a resposta) e **folhas** (que devolvem uma previsão). Vamos construí-la com o algoritmo **ID3**, que é relativamente simples e opera assim. Dados alguns exemplos rotulados e uma lista de atributos sobre os quais ramificar:

> **🔷 Conceito**
>
> 1. Se todos os dados têm o **mesmo rótulo**, crie uma folha que prevê aquele rótulo e pare.
> 2. Se a lista de atributos está **vazia** (não há mais perguntas a fazer), crie uma folha que prevê o rótulo mais comum e pare.
> 3. Caso contrário, particione os dados por **cada um** dos atributos.
> 4. Escolha a partição de **menor entropia**.
> 5. Adicione um nó de decisão baseado no atributo escolhido.
> 6. **Repita** dentro de cada subconjunto da partição, usando os atributos restantes.

Note o passo 6: cada ramo continua com **um atributo a menos**. Um atributo já usado não é reperguntado dentro do próprio ramo — dentro dele, todo mundo tem o mesmo valor daquele atributo, então perguntar de novo não separaria nada.

E note o que **não** está na lista: uma condição de parada. Os passos 1 e 2 param quando não sobrou dúvida ou quando não sobrou pergunta, e nenhum dos dois é um freio — os dois são o fim natural do crescimento. O jeito mais simples de conter uma árvore seria acrescentar um: profundidade máxima, mínimo de exemplos para dividir um nó, mínimo de exemplos numa folha. O ID3 não tem nenhum, e a [seção 14.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html) vai atacar o mesmo problema por um caminho completamente diferente — e depois medir os dois.

### Guloso quer dizer o quê

Esse é o que se chama de algoritmo **guloso**, porque a cada passo ele escolhe a opção imediatamente melhor.

> **❗ Importante**
>
> Dado um conjunto de dados, pode existir uma árvore melhor cujo **primeiro movimento pareça pior**. O ID3 não vai encontrá-la: ele escolhe pelo ganho imediato e nunca revê a escolha. Não há retrocesso, não há busca, não há "e se eu tivesse perguntado outra coisa primeiro".
>
> Isso não é um defeito da nossa implementação — é a definição do algoritmo, e é o preço de não resolver um problema computacionalmente muito difícil. A [seção 14.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/05-juntando-tudo.html) mostra um conjunto de dados de oito linhas em que essa escolha gulosa produz, comprovadamente, uma árvore maior do que a necessária.
>
> Em compensação, o ID3 é fácil de entender e de implementar, o que faz dele um bom ponto de partida para explorar árvores de decisão. E vale registrar que o `scikit-learn`, o `XGBoost` e todas as implementações sérias de árvore em uso hoje também são gulosas, pelo mesmo motivo.

### Percorrendo o algoritmo à mão

A [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/03-a-entropia-de-uma-particao.html) já executou à mão os primeiros passos deste algoritmo sobre os dados dos candidatos: `level` venceu a disputa da raiz com 0,694 bit; o grupo `Mid` saiu puro e virou folha na hora, pelo passo 1; e o grupo `Senior` foi resolvido por `tweets`, com entropia exatamente zero. Falta o terceiro ramo — e é nele que aparece uma coisa que ainda não vimos.

As funções e os dados são exatamente os de lá, então em vez de reescrevê-los importamos do pacote:

In [ ]:
from collections import Counter
from scratch.decision_trees import (partition_by, partition_entropy_by, inputs)

Os cinco juniores continuam misturados, então o passo 6 se aplica: repetimos tudo dentro do ramo, com os atributos que restaram. São três — `level` já foi gasto e não volta.

In [ ]:
junior_inputs = [entrada for entrada in inputs if entrada.level == 'Junior']

for key in ['lang', 'tweets', 'phd']:
    print(f"{key:8s} {partition_entropy_by(junior_inputs, key, 'did_well'):.6f}")

Mesma coisa que aconteceu com os sêniores, por outro atributo: dividir os juniores por `phd` dá **0,000000**. Conferindo diretamente:

In [ ]:
for valor, grupo in partition_by(junior_inputs, 'phd').items():
    print(f"phd={str(valor):5s} n={len(grupo)}  "
          f"{dict(Counter(c.did_well for c in grupo))}")

Entre os juniores, quem não tem doutorado se saiu bem — sempre; quem tem, se saiu mal — sempre. Os três ramos terminaram, e a árvore está completa:

In [ ]:
# Figura: A árvore de decisão para contratação, deduzida à mão. Caixas claras são perguntas; verdes e vermelhas, previsões.
from matplotlib import pyplot as plt

# (pergunta, {valor: subárvore}); uma folha é a string da previsão.
arvore = ("level?", {
    "Senior": ("tweets?", {"False": "NÃO CONTRATAR", "True": "CONTRATAR!"}),
    "Mid": "CONTRATAR!",
    "Junior": ("phd?", {"False": "CONTRATAR!", "True": "NÃO CONTRATAR"}),
})

def folhas(t):
    return 1 if isinstance(t, str) else sum(folhas(s) for s in t[1].values())

def altura(t):
    return 0 if isinstance(t, str) else 1 + max(altura(s) for s in t[1].values())

def desenha(ax, t, x0, x1, prof, h):
    y, x = h - prof, (x0 + x1) / 2
    if isinstance(t, str):
        cor, ec = ('#d7ecd9', '#4a8f55') if t == "CONTRATAR!" else ('#f6d9d9', '#b55a5a')
        ax.text(x, y, t, ha='center', va='center', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.4', fc=cor, ec=ec))
        return x, y
    ax.text(x, y, t[0], ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', fc='#f6ead0', ec='#b58a4a'))
    total, ini = folhas(t), x0
    for valor, sub in t[1].items():
        larg = (x1 - x0) * folhas(sub) / total
        xs, ys = desenha(ax, sub, ini, ini + larg, prof + 1, h)
        ax.annotate("", xy=(xs, ys + 0.18), xytext=(x, y - 0.18),
                    arrowprops=dict(arrowstyle='->', color='#888', lw=1))
        ax.text((x + xs) / 2, (y + ys) / 2, valor, fontsize=8, color='#444',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.9))
        ini += larg
    return x, y

h = altura(arvore)
fig, ax = plt.subplots(figsize=(9, 1.3 * h + 0.8))
desenha(ax, arvore, 0, 1, 0, h)
ax.set_xlim(-0.08, 1.08)
ax.set_ylim(-0.35, h + 0.35)
ax.axis('off')
plt.tight_layout()
plt.show()

Duas perguntas bastam para classificar qualquer candidato, e frequentemente uma só. Repare no que essa figura permite: você pode discordar dela. Pode dizer ao vice-presidente que a regra dos sêniores está apoiada em cinco pessoas, ou que `phd` prever fracasso entre juniores é provavelmente ruído. Não dá para fazer isso com um vetor de coeficientes ou com uma lista de vizinhos — é essa a moeda que a árvore paga a mais.

### A pergunta que este ID3 não sabe fazer

O `partition_by` da [seção 14.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/03-a-entropia-de-uma-particao.html) abre **um ramo por valor distinto** do atributo. Sobre `level`, isso dá três ramos e é exatamente o que se quer. Sobre uma coluna contínua — salário, temperatura, idade em dias —, dá um ramo por valor: duzentos salários diferentes viram duzentos ramos de um exemplo cada, todos de entropia zero. É a coluna de matrícula daquela seção outra vez, agora **sem que ninguém tenha acrescentado uma coluna boba**: basta a coluna ser contínua. É por isso que as implementações de verdade não perguntam "qual é o valor?", e sim "**o valor está acima de tal limiar?**" — `salário > 60.000?` —, escolhendo o limiar junto com o atributo, entre os cortes possíveis, pelo mesmo critério de entropia. Divisão binária, dois ramos, uma fatia grande dos dados em cada. É o que faz o CART, e é a diferença estrutural entre a árvore do `scikit-learn` e a nossa.

Duas consequências disso valem registrar. A primeira separa a árvore de todos os modelos dos capítulos [11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html) a [13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html): como toda pergunta é sobre a **ordem** dos valores de uma coluna, e reescalonar preserva a ordem, **medir salário em reais ou em desvios padrão é rigorosamente indiferente para uma árvore**. Não há normalização a fazer, e a fronteira que sai é feita de degraus paralelos aos eixos, em vez do hiperplano inclinado que o produto escalar produzia. A segunda é menos confortável: a árvore que sabe perguntar por limiar também sabe tratar um atributo faltante — mandando o exemplo pelos dois lados, com peso — e a nossa não sabe nem uma coisa nem outra.

> **💡 Dica — Na prática: a biblioteca é gulosa igual — e o que ela acrescenta**
>
> O `DecisionTreeClassifier` do `scikit-learn` é guloso pelo mesmo motivo que o nosso: encontrar a árvore ótima é intratável. O que a biblioteca acrescenta não é busca, é **controle de crescimento** — porque a árvore que este capítulo constrói cresce até a pureza total, e isso é sobreajuste garantido:
>
> ```python
> from sklearn.tree import DecisionTreeClassifier
>
> DecisionTreeClassifier(
>     max_depth=3,          # não pergunte mais que 3 vezes
>     min_samples_split=10, # não divida um nó com menos de 10 exemplos
>     min_samples_leaf=5,   # não crie folha com menos de 5 exemplos
>     ccp_alpha=0.01,       # poda: remova ramos que não pagam sua complexidade
> )
> ```
>
> Os três primeiros são **pré-poda**: interrompem o crescimento antes que ele aconteça. São baratos e é o que se usa na prática, mas são gulosos também — podem cortar um ramo que só se revelaria útil um nível abaixo.
>
> O `ccp_alpha` é **pós-poda** (*minimal cost-complexity pruning*): cresce a árvore inteira e depois remove os ramos cujo ganho não compensa o tamanho, controlado por um parâmetro $\alpha$ que penaliza o número de folhas. É a mesma forma de regularização que a [seção 12.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/08-regularizacao.html) aplicou aos coeficientes de uma regressão — penalizar a complexidade do modelo dentro do próprio critério de ajuste —, só que aqui a complexidade se conta em folhas em vez de em magnitude de parâmetros.
>
> O algoritmo deste capítulo não tem nenhum dos dois. Ele para apenas quando os rótulos ficam puros ou quando os atributos acabam, que são os passos 1 e 2 do ID3. A [seção 14.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html) ataca o mesmo problema por um caminho completamente diferente — em vez de encolher a árvore, construir muitas árvores grandes e fazê-las votar — e depois mede os dois caminhos lado a lado. O resultado não é o que se espera.

## Juntando Tudo

> **📌 Nota**
>
> Esta seção corresponde a *Putting It All Together*, do capítulo 17 de Grus (2019).

Agora que vimos como o algoritmo funciona, queremos implementá-lo de forma mais geral — não só para os candidatos a emprego. Para isso é preciso decidir como **representar** uma árvore.

Vamos usar a representação mais leve possível. Uma árvore é uma de duas coisas:

- uma **`Leaf`** (que prevê um único valor), ou
- um **`Split`** (que contém um atributo sobre o qual dividir, as subárvores para valores específicos daquele atributo e, possivelmente, um valor padrão para usar quando aparecer um valor desconhecido).

In [ ]:
from typing import NamedTuple, Union, Any

class Leaf(NamedTuple):
    value: Any

class Split(NamedTuple):
    attribute: str
    subtrees: dict
    default_value: Any = None

DecisionTree = Union[Leaf, Split]

São onze linhas, e nelas cabe o modelo inteiro. Compare com o que os capítulos anteriores precisavam guardar: um vetor `beta` de coeficientes que só significa alguma coisa junto com a documentação de qual coluna é qual, ou — no caso do k-vizinhos — o conjunto de dados inteiro.

Com essa representação, a árvore de contratação da [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/04-criando-uma-arvore.html) fica assim:

In [ ]:
hiring_tree = Split('level', {                # primeiro, considere "level"
    'Junior': Split('phd', {                  # se level é "Junior", olhe "phd"
        False: Leaf(True),                    #   se "phd" é False, preveja True
        True: Leaf(False)                     #   se "phd" é True, preveja False
    }),
    'Mid': Leaf(True),                        # se level é "Mid", preveja True
    'Senior': Split('tweets', {               # se level é "Senior", olhe "tweets"
        False: Leaf(False),                   #   se "tweets" é False, preveja False
        True: Leaf(True)                      #   se "tweets" é True, preveja True
    })
})

Isso é a figura da seção anterior, escrita em Python. Ela cabe na tela, e um leitor que nunca ouviu falar de entropia consegue conferir se concorda com ela.

### O valor que ninguém previu

Resta a pergunta do que fazer diante de um valor de atributo **inesperado** (ou faltante). O que a nossa árvore deveria fazer se encontrasse um candidato cujo `level` fosse `Intern`? Esse caso é tratado preenchendo o atributo `default_value` com o rótulo mais comum daquele nó.

Dada essa representação, dá para classificar uma entrada:

In [ ]:
def classify(tree: DecisionTree, input: Any) -> Any:
    """classifica a entrada usando a árvore de decisão dada"""

    # Se este é um nó folha, devolve o seu valor
    if isinstance(tree, Leaf):
        return tree.value

    # Caso contrário, esta árvore consiste de um atributo sobre o qual dividir
    # e de um dicionário cujas chaves são valores daquele atributo
    # e cujos valores são subárvores a considerar em seguida
    subtree_key = getattr(input, tree.attribute)

    if subtree_key not in tree.subtrees:   # Se não há subárvore para a chave,
        return tree.default_value          # devolve o valor padrão.

    subtree = tree.subtrees[subtree_key]   # Escolhe a subárvore apropriada
    return classify(subtree, input)        # e a usa para classificar a entrada.

### Construindo a árvore a partir dos dados

Falta só montar a representação a partir dos dados de treino — ou seja, escrever em código os seis passos do ID3:

In [ ]:
from typing import List
from collections import Counter
from scratch.decision_trees import (partition_by, partition_entropy_by,
                                    Candidate, inputs)

> **📌 Nota**
>
> `Candidate`, `inputs`, `partition_by` e `partition_entropy_by` são exatamente os da [seção 14.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/03-a-entropia-de-uma-particao.html). As funções que **são** a lição desta seção — `classify` e `build_tree_id3` — estão escritas aqui, à vista.

In [ ]:
def build_tree_id3(inputs: List[Any],
                   split_attributes: List[str],
                   target_attribute: str) -> DecisionTree:
    # Conta os rótulos do alvo
    label_counts = Counter(getattr(input, target_attribute)
                           for input in inputs)
    most_common_label = label_counts.most_common(1)[0][0]

    # Se há um único rótulo, preveja-o
    if len(label_counts) == 1:
        return Leaf(most_common_label)

    # Se não sobraram atributos para dividir, devolva o rótulo majoritário
    if not split_attributes:
        return Leaf(most_common_label)

    # Caso contrário, divida pelo melhor atributo

    def split_entropy(attribute: str) -> float:
        """Função auxiliar para encontrar o melhor atributo"""
        return partition_entropy_by(inputs, attribute, target_attribute)

    best_attribute = min(split_attributes, key=split_entropy)

    partitions = partition_by(inputs, best_attribute)
    new_attributes = [a for a in split_attributes if a != best_attribute]

    # Constrói as subárvores recursivamente
    subtrees = {attribute_value: build_tree_id3(subset,
                                                new_attributes,
                                                target_attribute)
                for attribute_value, subset in partitions.items()}

    return Split(best_attribute, subtrees, default_value=most_common_label)

Cada uma das seis regras do ID3 aparece aí, na ordem: os dois `if` que devolvem `Leaf` são os passos 1 e 2, o `min(split_attributes, key=split_entropy)` é o passo 4, e a compreensão de dicionário no fim é o passo 6. O passo 3 — particionar por cada atributo — está escondido dentro do `key=`, que chama `split_entropy` uma vez para cada candidato a atributo.

Vamos construir a árvore com os quatro atributos e olhar o que saiu:

In [ ]:
tree = build_tree_id3(inputs,
                      ['level', 'lang', 'tweets', 'phd'],
                      'did_well')

def imprime(arvore: DecisionTree, nivel: int = 0, prefixo: str = "") -> None:
    espaco = "    " * nivel
    if isinstance(arvore, Leaf):
        print(f"{espaco}{prefixo}=> {arvore.value}")
    else:
        print(f"{espaco}{prefixo}{arvore.attribute}?")
        for valor, sub in arvore.subtrees.items():
            imprime(sub, nivel + 1, f"{valor}: ")

imprime(tree)

É a mesma árvore da figura da [seção 14.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/04-criando-uma-arvore.html), agora deduzida pelo código em vez de por nós. As sete linhas acima são o modelo inteiro — não uma visualização aproximada dele.

### Prevendo

Na árvore que construímos, cada folha consiste inteiramente de entradas `True` ou inteiramente de entradas `False`. Isso significa que ela prevê **perfeitamente** o conjunto de treino:

In [ ]:
acertos = sum(1 for c in inputs if classify(tree, c) == c.did_well)
print(f"acertos no conjunto de treino: {acertos}/{len(inputs)}")

Mas também dá para aplicá-la a dados novos, que não estavam no treino:

In [ ]:
# Deve prever True
assert classify(tree, Candidate("Junior", "Java", True, False))

# Deve prever False
assert not classify(tree, Candidate("Junior", "Java", True, True))

for candidato in [Candidate("Junior", "Java", True, False),
                  Candidate("Junior", "Java", True, True)]:
    print(f"{candidato} -> {classify(tree, candidato)}")

Os dois casos fazem sentido: os dois são juniores, e a árvore pergunta `phd` para juniores. Sem doutorado, `True`; com doutorado, `False`.

E também dá para aplicá-la a dados com valores inesperados:

In [ ]:
# Prevê True — e o callout abaixo é sobre o motivo
assert classify(tree, Candidate("Intern", "Java", True, True))

estagiario = Candidate("Intern", "Java", True, True)
print(f"{estagiario} -> {classify(tree, estagiario)}")

> **⚠️ Atenção — O modelo respondeu com confiança sobre algo que nunca viu**
>
> `Intern` **não aparece em nenhuma das 14 linhas de treino**. Nenhum estagiário foi entrevistado, nenhum estagiário foi rotulado, e o algoritmo não tem a menor informação sobre como estagiários se saem em entrevistas.
>
> Ainda assim, `classify` devolveu `True`, sem hesitar e sem sinalizar nada. O mecanismo está à vista no código: `subtree_key not in tree.subtrees` é verdadeiro, e a função devolve `tree.default_value`, que `build_tree_id3` preencheu com `most_common_label` — o rótulo mais comum naquele nó, que na raiz é `True`, porque 9 dos 14 candidatos se saíram bem.
>
> Ou seja: a resposta para "como se sai um estagiário?" é "como se sai a maioria das pessoas que **não** são estagiárias". É um padrão razoável de implementação e uma resposta péssima de modelo, e a diferença entre as duas coisas some completamente na saída, que é um `True` idêntico ao dos outros dois casos.
>
> Este é um defeito da **interface**, não do algoritmo. Um `classify` que devolvesse `(previsão, viu_esse_valor)` — ou que levantasse uma exceção diante de um valor não visto — daria ao chamador a chance de tratar o caso. Vale a pena guardar a lição de forma geral: **a maioria dos modelos, na maioria das bibliotecas, não distingue "eu sei" de "eu chutei o mais comum"**, e a previsão sai com a mesma cara nos dois casos. Se essa distinção importa para o seu problema, ela precisa ser construída por fora.

### O guloso não encontra a menor árvore

A [seção 14.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/04-criando-uma-arvore.html) afirmou que o ID3 pode produzir uma árvore maior do que a necessária. Agora que `build_tree_id3` existe, dá para mostrar.

Considere oito exemplos com três atributos, em que o alvo é exatamente `a != b` — o atributo `c` está lá só para atrapalhar:

In [ ]:
from typing import Optional

class Exemplo(NamedTuple):
    a: int
    b: int
    c: int
    alvo: Optional[bool] = None

dados = [Exemplo(0, 0, 1, False), Exemplo(0, 1, 1, True),
         Exemplo(1, 0, 1, True),  Exemplo(1, 1, 0, False),
         Exemplo(0, 0, 0, False), Exemplo(0, 1, 1, True),
         Exemplo(1, 0, 0, True),  Exemplo(1, 1, 0, False)]

for key in ['a', 'b', 'c']:
    print(f"{key} {partition_entropy_by(dados, key, 'alvo'):.6f}")

Sozinhos, nem `a` nem `b` reduzem a incerteza em nada — cada um deles deixa os dois lados meio a meio, entropia 1,000000. Só o par `(a, b)` decide o alvo, e o critério guloso, que olha um atributo de cada vez, é **cego para isso**. Já `c` oferece uma redução pequena mas real, então é ele que vence o primeiro passo.

Construímos duas árvores: uma com os três atributos, como o algoritmo faria, e outra escondendo `c` dele.

In [ ]:
def folhas(t: DecisionTree) -> int:
    return 1 if isinstance(t, Leaf) else sum(folhas(s) for s in t.subtrees.values())

guloso = build_tree_id3(dados, ['a', 'b', 'c'], 'alvo')
enxuta = build_tree_id3(dados, ['a', 'b'], 'alvo')

for nome, t in [("com a, b, c", guloso), ("só com a, b", enxuta)]:
    certos = sum(1 for d in dados if classify(t, d) == d.alvo)
    print(f"{nome:12s} raiz={t.attribute:2s} folhas={folhas(t)}  "
          f"acertos={certos}/{len(dados)}")

As duas acertam os oito exemplos de treino. A gulosa gasta **seis folhas** para isso; escondendo `c` do algoritmo, quatro bastam. Em vez de acreditar na contagem, vale olhar:

In [ ]:
for nome, t in [("com a, b, c", guloso), ("só com a, b", enxuta)]:
    print(f"--- {nome} ---")
    imprime(t)

A árvore enxuta **é** a regra `a != b`, escrita como árvore, e nada mais. A gulosa começa perguntando `c`, que não tem relação nenhuma com o alvo, e a partir daí é obrigada a reconstruir a regra inteira **duas vezes**, uma dentro de cada ramo de `c` — e mesmo assim não termina o serviço, porque nem toda combinação de `(a, b)` apareceu dentro de cada ramo.

Isso tem consequência medível, e ela não aparece no treino. Os oito exemplos não cobrem as oito combinações possíveis de `(a, b, c)`: algumas se repetem e duas nunca aparecem. Testando as duas árvores contra as oito, com o alvo verdadeiro `a != b`:

In [ ]:
import itertools

todas = [Exemplo(a, b, c, a != b)
         for a, b, c in itertools.product([0, 1], repeat=3)]

for nome, t in [("com a, b, c", guloso), ("só com a, b", enxuta)]:
    erros = [(d.a, d.b, d.c) for d in todas if classify(t, d) != d.alvo]
    print(f"{nome:12s} acertos={8 - len(erros)}/8  erra em {erros}")

A enxuta acerta as oito. A gulosa erra duas — e são exatamente as duas que não estavam no treino. Ela decorou os ramos de `c` que viu e não tem o que dizer sobre os que não viu, que é o mecanismo do sobreajuste na sua forma mais nua.

> **⚠️ Atenção**
>
> Uma árvore com mais folhas para o mesmo desempenho no treino é uma árvore que particionou os dados mais finamente sem necessidade, e essa é precisamente a receita para generalizar pior. Aqui o "pior" foi de 8/8 para 6/8, sobre um problema de três atributos binários.
>
> E note que "esconder `c`" não é uma opção disponível na vida real — se soubéssemos de antemão quais atributos atrapalham, não precisaríamos do algoritmo.

> **❗ Importante**
>
> Como o objetivo aqui era demonstrar **como** construir uma árvore, nós a construímos sobre o conjunto de dados inteiro. Como sempre, se estivéssemos realmente tentando criar um bom modelo para alguma coisa, teríamos coletado mais dados e os dividido em subconjuntos de treino, validação e teste — como a [seção 8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html) insistiu.
>
> Sem essa divisão, os `14/14` acima não medem absolutamente nada sobre a qualidade do modelo. Eles medem apenas que o algoritmo fez o que promete fazer: dividir até a pureza.

> **💡 Dica — Na prática: `DecisionTreeClassifier`**
>
> O equivalente de tudo isso na biblioteca:
>
> ```python
> from sklearn.tree import DecisionTreeClassifier, export_text
> from sklearn.preprocessing import OrdinalEncoder
>
> X = OrdinalEncoder().fit_transform([[c.level, c.lang, c.tweets, c.phd]
>                                     for c in inputs])
> y = [c.did_well for c in inputs]
>
> modelo = DecisionTreeClassifier(criterion='entropy', random_state=0).fit(X, y)
>
> print(export_text(modelo, feature_names=['level', 'lang', 'tweets', 'phd']))
> ```
>
> O `export_text` imprime a árvore treinada em texto indentado, e `sklearn.tree.plot_tree` a desenha — as duas coisas que o nosso `imprime` e a figura da [seção 14.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/04-criando-uma-arvore.html) fazem. É raro uma biblioteca de aprendizado de máquina oferecer isso, e não é generosidade: é que, para uma árvore, imprimir o modelo é possível.
>
> Três diferenças que importam ao comparar as saídas:
>
> - **A árvore é binária.** O `scikit-learn` implementa CART, que divide sempre em dois ramos por um limiar numérico (`level <= 1.5`), enquanto o nosso ID3 abre um ramo por valor (`Senior`, `Mid`, `Junior` de uma vez). A árvore da biblioteca sobre estes dados terá mais níveis para expressar a mesma regra.
> - **Os atributos precisam ser números.** Daí o `OrdinalEncoder` acima. E ele introduz uma ordem que não existe nos dados: codificar `Junior=0, Mid=1, Senior=2` faz o corte `<= 1.5` significar "Junior ou Mid", um agrupamento que ninguém pediu. Para atributos categóricos sem ordem natural, o usual é `OneHotEncoder`, ao custo de uma coluna por valor.
> - **Não há `default_value`.** O caso `Intern` sequer chega ao modelo: o `OrdinalEncoder` levanta um erro diante de uma categoria não vista, a menos que você configure `handle_unknown='use_encoded_value'`. É uma falha mais barulhenta que a nossa — e, dado o callout acima, provavelmente melhor.

## Florestas Aleatórias

> **📌 Nota**
>
> Esta seção corresponde a *Random Forests*, do capítulo 17 de Grus (2019).

A [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/05-juntando-tudo.html) terminou com a árvore acertando 14 dos 14 candidatos que usou para se construir. Dado o quanto uma árvore de decisão consegue se ajustar aos próprios dados de treino, não surpreende que ela tenha **tendência ao sobreajuste**.

Vale enxergar por que a tendência é estrutural, e não azar. A regra de parada do ID3 é "pare quando os rótulos ficarem puros ou quando os atributos acabarem". Com atributos suficientes, a primeira condição sempre chega antes da segunda — a árvore simplesmente continua perguntando até cada folha conter uma única classe, ainda que para isso precise isolar exemplos um a um. O [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) chamou isso de sobreajuste; a árvore é o modelo deste livro que chega lá mais rápido, porque nada no algoritmo a impede.

Uma forma de evitá-lo é a **floresta aleatória**: construir várias árvores de decisão e combinar as saídas. Se forem árvores de classificação, deixamos que votem; se forem de regressão, tiramos a média das previsões.

Mas o nosso processo de construção é **determinístico** — os mesmos dados produzem sempre a mesma árvore. De onde viriam árvores diferentes?

### Fonte de aleatoriedade 1: reamostrar os dados

A primeira peça é reamostrar os dados. Em vez de treinar cada árvore com todas as `inputs` do conjunto de treino, treinamos cada árvore com o resultado de `bootstrap_sample(inputs)`.

Esse é o mesmo `bootstrap_sample` do [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/06-digressao-o-bootstrap.html) — sortear, com reposição, tantos elementos quanto a amostra original tem. Lá o objetivo era **medir incerteza**: reamostrava-se para ver o quanto uma estatística balançava. Aqui o objetivo é outro, reamostrar para obter conjuntos de treino diferentes, cada um produzindo uma árvore diferente. A mecânica é idêntica; o uso, não — e é o tipo de reaproveitamento que compensa reconhecer, porque a mesma linha de código serve para duas coisas sem relação uma com a outra.

In [ ]:
import random
from typing import List, TypeVar

X = TypeVar('X')

def bootstrap_sample(data: List[X]) -> List[X]:
    """sorteia len(data) elementos com reposição"""
    return [random.choice(data) for _ in data]

Como cada árvore é construída com dados diferentes, cada árvore sai diferente das outras.

Há um benefício de brinde nisso. Uma amostra bootstrap de tamanho $n$ deixa de fora, em média, cerca de 37% dos exemplos originais — a chance de um exemplo específico nunca ser sorteado em $n$ tentativas é $(1 - 1/n)^n$, que converge para $1/e \approx 0{,}368$. Esses exemplos não sorteados são chamados **out-of-bag**, e são um conjunto de teste honesto para aquela árvore em particular: ela nunca os viu. Isso permite estimar o desempenho da floresta **usando todos os dados como treino**, sem separar um conjunto de teste — o que, em conjuntos pequenos, é uma vantagem real.

A técnica de treinar vários modelos em reamostras bootstrap e combinar as saídas chama-se **bootstrap aggregating**, ou **bagging**.

### Fonte de aleatoriedade 2: sortear os atributos

A segunda peça muda a forma de escolher o `best_attribute` sobre o qual dividir. Em vez de olhar **todos** os atributos restantes, sorteamos primeiro um subconjunto deles e dividimos pelo melhor desse subconjunto:

In [ ]:
from typing import Any
from collections import Counter
from scratch.decision_trees import (partition_by, partition_entropy_by,
                                    build_tree_id3, classify,
                                    Leaf, Split, DecisionTree, Candidate, inputs)

In [ ]:
def build_tree_forest(inputs: List[Any],
                      split_attributes: List[str],
                      target_attribute: str,
                      num_split_candidates: int = 2) -> DecisionTree:
    """Igual ao build_tree_id3, exceto por sortear os atributos candidatos."""
    label_counts = Counter(getattr(input, target_attribute) for input in inputs)
    most_common_label = label_counts.most_common(1)[0][0]

    if len(label_counts) == 1:
        return Leaf(most_common_label)
    if not split_attributes:
        return Leaf(most_common_label)

    def split_entropy(attribute: str) -> float:
        return partition_entropy_by(inputs, attribute, target_attribute)

    # se já há poucos candidatos a divisão, olhe todos eles
    if len(split_attributes) <= num_split_candidates:
        sampled_split_candidates = split_attributes
    # caso contrário, sorteie uma amostra
    else:
        sampled_split_candidates = random.sample(split_attributes,
                                                 num_split_candidates)

    # agora escolha o melhor atributo apenas entre esses candidatos
    best_attribute = min(sampled_split_candidates, key=split_entropy)

    partitions = partition_by(inputs, best_attribute)
    new_attributes = [a for a in split_attributes if a != best_attribute]

    subtrees = {attribute_value: build_tree_forest(subset, new_attributes,
                                                  target_attribute,
                                                  num_split_candidates)
                for attribute_value, subset in partitions.items()}

    return Split(best_attribute, subtrees, default_value=most_common_label)

A diferença em relação ao `build_tree_id3` da [seção 14.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/05-juntando-tudo.html) cabe em cinco linhas — o `if`/`else` que reduz `split_attributes` a `sampled_split_candidates` antes do `min`. Todo o resto é idêntico.

E a combinação das saídas é uma votação simples:

In [ ]:
def forest_classify(trees: List[DecisionTree], input: Any) -> Any:
    votes = [classify(tree, input) for tree in trees]
    return Counter(votes).most_common(1)[0][0]

### Uma floresta sobre os candidatos

Com as duas peças, cem árvores:

In [ ]:
atributos = ['level', 'lang', 'tweets', 'phd']

random.seed(12)
floresta = [build_tree_forest(bootstrap_sample(inputs), atributos, 'did_well')
            for _ in range(100)]

raizes = Counter(t.attribute if isinstance(t, Split) else '(só uma folha)'
                 for t in floresta)
for atributo, n in raizes.most_common():
    print(f"{atributo:15s} {n:3d} árvore{'s' if n > 1 else ''}")

> **⚠️ Atenção — A semente não é opcional**
>
> `random.seed(12)` está ali porque tudo nesta seção é aleatório: as reamostras, os subconjuntos de atributos e, por consequência, as contagens acima. Sem a semente, cada execução constrói uma floresta diferente e devolve outros números — e aí ninguém consegue repetir o seu resultado, nem você mesmo no dia seguinte. Um experimento que não se repete não é uma medição, é uma anedota; a semente é o que transforma um em outro.
>
> Repare em qual gerador está sendo semeado: o `random` da biblioteca padrão, que é o que este código usa. `random.seed` não tem efeito nenhum sobre o `numpy.random`, que tem semente própria — semear um e sortear do outro é um jeito silencioso de achar que fixou a aleatoriedade sem ter fixado.

Lembre-se de que a árvore determinística da seção anterior dividia por `level` na raiz, sempre. Na floresta, `level` só é a raiz de uma fração das árvores; nas outras, a raiz é o atributo que por acaso caiu no sorteio e venceu localmente. Uma das cem nem chegou a ser uma árvore: a reamostra que ela recebeu, por sorte do bootstrap, continha candidatos de um único rótulo, e o passo 1 do ID3 devolveu uma folha na hora.

Como fica a votação nos três candidatos da seção anterior?

In [ ]:
testes = [Candidate("Junior", "Java", True, False),
          Candidate("Junior", "Java", True, True),
          Candidate("Intern", "Java", True, True)]

for candidato in testes:
    votos = Counter(classify(t, candidato) for t in floresta)
    print(f"{candidato.level:7s} phd={str(candidato.phd):5s} -> "
          f"{forest_classify(floresta, candidato)}   "
          f"(True: {votos[True]}, False: {votos[False]})")

As três previsões coincidem com as da árvore única, mas agora vêm com algo que a árvore única não dava: uma **contagem de votos**. Uma decisão de 78 a 22 e uma de 51 a 49 saem com a mesma cara num classificador comum; aqui a diferença fica visível. Repare, contudo, que a maioria confortável no caso `Intern` não significa que a floresta saiba alguma coisa sobre estagiários — significa apenas que a maioria das árvores respondeu com o seu próprio `default_value`, pelo mecanismo que o callout da [seção 14.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/05-juntando-tudo.html) descreveu.

> **❗ Importante**
>
> Catorze exemplos não permitem **medir** se a floresta é melhor que a árvore. Com esse tamanho, qualquer comparação de acurácia oscila mais em função de qual exemplo caiu em qual lado do que em função do método, e uma medição dessas diria mais sobre o sorteio do que sobre o algoritmo.
>
> A [seção 8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html) mostra o tamanho desse efeito em números: numa base pequena e desbalanceada, a quantidade de positivos que cai no conjunto de teste varia de 0 a 6 só em função da semente. Uma medição feita sobre poucas repetições é um chute com aparência de número. Então vamos medir direito, num conjunto grande o bastante para a resposta significar alguma coisa.

### Medindo o que a floresta compra

Vamos fabricar um problema com estrutura conhecida. Oito atributos categóricos, cada um com três valores possíveis; o rótulo depende **apenas** dos dois primeiros, e 15% dos rótulos de treino são invertidos, para simular ruído de medição. Os outros seis atributos são puro ruído.

In [ ]:
from typing import NamedTuple, Optional

class Linha(NamedTuple):
    a0: str
    a1: str
    a2: str
    a3: str
    a4: str
    a5: str
    a6: str
    a7: str
    alvo: Optional[bool] = None

ATRIBUTOS = [f"a{i}" for i in range(8)]
VALORES = ['x', 'y', 'z']

def gera(n: int, ruido: float = 0.15) -> List[Linha]:
    linhas = []
    for _ in range(n):
        valores = [random.choice(VALORES) for _ in range(8)]
        alvo = (valores[0] == 'x') or (valores[1] == 'y')   # a regra verdadeira
        if random.random() < ruido:
            alvo = not alvo                                  # rótulo corrompido
        linhas.append(Linha(*valores, alvo))
    return linhas

O conjunto de treino tem 200 linhas com 15% de ruído; o de teste tem 1.000 linhas **sem** ruído, para medir o quanto cada modelo recuperou a regra verdadeira em vez de decorar as corrupções. Repetimos o experimento inteiro 20 vezes, com dados novos a cada vez.

Falta uma peça antes de comparar. A [seção 14.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/04-criando-uma-arvore.html) disse que o remédio mais simples contra o crescimento sem freio seria uma condição de parada, e não a implementou. É uma linha dentro do `build_tree_id3`:

In [ ]:
def build_tree_podada(inputs: List[Any],
                      split_attributes: List[str],
                      target_attribute: str,
                      max_depth: int,
                      profundidade: int = 0) -> DecisionTree:
    """Igual ao build_tree_id3, exceto por virar folha ao atingir max_depth."""
    label_counts = Counter(getattr(input, target_attribute) for input in inputs)
    most_common_label = label_counts.most_common(1)[0][0]

    if len(label_counts) == 1:
        return Leaf(most_common_label)
    if not split_attributes:
        return Leaf(most_common_label)
    if profundidade >= max_depth:            # a única linha nova
        return Leaf(most_common_label)

    def split_entropy(attribute: str) -> float:
        return partition_entropy_by(inputs, attribute, target_attribute)

    best_attribute = min(split_attributes, key=split_entropy)

    partitions = partition_by(inputs, best_attribute)
    new_attributes = [a for a in split_attributes if a != best_attribute]

    subtrees = {valor: build_tree_podada(sub, new_attributes, target_attribute,
                                         max_depth, profundidade + 1)
                for valor, sub in partitions.items()}

    return Split(best_attribute, subtrees, default_value=most_common_label)

É o `max_depth` do `DecisionTreeClassifier`, escrito à mão. Com ele, dá para comparar as duas respostas ao sobreajuste — encolher a árvore e multiplicar as árvores — na mesma medição, em vez de acreditar em uma delas:

In [ ]:
def acuracia(prever, dados) -> float:
    return sum(1 for linha in dados if prever(linha) == linha.alvo) / len(dados)

random.seed(0)

resultados = {'árvore única': [], 'árvore prof. 1': [], 'árvore prof. 2': [],
              'árvore prof. 3': [], 'só bagging': [],
              'floresta (2 de 8)': [], 'floresta (3 de 8)': []}
no_proprio_treino = {'árvore única': [], 'árvore prof. 2': []}

for _ in range(20):
    treino = gera(200)
    teste = gera(1000, ruido=0.0)

    arvore = build_tree_id3(treino, ATRIBUTOS, 'alvo')
    no_proprio_treino['árvore única'].append(
        acuracia(lambda linha: classify(arvore, linha), treino))
    resultados['árvore única'].append(
        acuracia(lambda linha: classify(arvore, linha), teste))

    for d in (1, 2, 3):
        podada = build_tree_podada(treino, ATRIBUTOS, 'alvo', d)
        resultados[f'árvore prof. {d}'].append(
            acuracia(lambda linha: classify(podada, linha), teste))
        if d == 2:
            no_proprio_treino['árvore prof. 2'].append(
                acuracia(lambda linha: classify(podada, linha), treino))

    bagging = [build_tree_id3(bootstrap_sample(treino), ATRIBUTOS, 'alvo')
               for _ in range(50)]
    resultados['só bagging'].append(
        acuracia(lambda linha: forest_classify(bagging, linha), teste))

    for k in (2, 3):
        floresta_k = [build_tree_forest(bootstrap_sample(treino), ATRIBUTOS,
                                        'alvo', k)
                      for _ in range(50)]
        resultados[f'floresta ({k} de 8)'].append(
            acuracia(lambda linha: forest_classify(floresta_k, linha), teste))

for nome, valores in no_proprio_treino.items():
    print(f"{nome:18s} NO TREINO  média={sum(valores) / len(valores):.4f}")
print()
for nome, valores in resultados.items():
    media = sum(valores) / len(valores)
    print(f"{nome:18s} no teste   média={media:.4f}  pior={min(valores):.3f}  "
          f"melhor={max(valores):.3f}")

Comece pela árvore sem poda, que é o modelo da [seção 14.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/05-juntando-tudo.html). Ela acerta **99,7%** dos exemplos que usou para se construir — inclusive os 15% cujo rótulo foi invertido de propósito, que ela só consegue acertar decorando-os um a um, com perguntas sobre os seis atributos de puro ruído. Sobre dados novos, cai para **82,4%**. A distância entre esses dois números é o sobreajuste, medido.

(Por que 99,7% e não 100%? Porque com 200 sorteios sobre $3^8 = 6.561$ combinações possíveis de atributos, de vez em quando duas linhas idênticas recebem rótulos diferentes — e aí nem uma árvore infinitamente profunda consegue separá-las. O passo 2 do ID3 devolve o rótulo majoritário e erra o outro.)

Os três comitês fazem o que se espera deles: 90,8% só com o bagging, 91,7% e **92,8%** com o sorteio de atributos — até dez pontos percentuais acima da árvore sozinha, e sem que ninguém precise dizer nada ao algoritmo sobre a estrutura do problema.

E aí vem a linha que estraga o final feliz. A árvore de **profundidade 2** — duas perguntas, nove folhas, um modelo que cabe num cartão — acerta **100,00%** do conjunto de teste, nas 20 repetições, sem uma única exceção. Nenhuma floresta chega perto.

Repare no que ela faz no treino: **84,95%**, muito *pior* que os 99,7% da árvore sem poda. Não é acaso — 85% é exatamente a fração de rótulos que não foram corrompidos. A árvore rasa acerta todo o sinal e erra todo o ruído, que é o comportamento correto, e a árvore profunda "melhora" esse número decorando as corrupções uma a uma. **O modelo pior no treino é o melhor no teste, e a diferença é grande.**

Não é sorte: é a estrutura do problema. A regra verdadeira é `(a0 == 'x') or (a1 == 'y')` — dois atributos, duas perguntas. Com profundidade 2, a árvore pergunta exatamente duas vezes; o critério de entropia escolhe `a0` e `a1`, que são os únicos com sinal; e o limite a impede de perguntar sobre o ruído, que é como a árvore sem poda decora. Ela não tem para onde crescer, e isso aqui é uma vantagem.

A profundidade importa nos dois sentidos, e a tabela mostra a curva inteira: com **uma** pergunta só, 77,3% — não dá para expressar um "ou" entre dois atributos com uma pergunta; com **três**, 94,6% — sobra uma pergunta, e ela é gasta no ruído; sem limite, 82,4%. O desempenho sobe até o modelo ter exatamente a capacidade da regra verdadeira e desce a partir dali. É o U do compromisso viés-variância da [seção 8.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/05-vies-e-variancia.html), desenhado por um modelo que não tem um único parâmetro contínuo.

> **🔷 Conceito**
>
> As duas fontes de aleatoriedade contribuem, e contribuem por motivos diferentes.
>
> O **bagging** sozinho já resolve boa parte do problema: cada árvore decora um conjunto diferente de ruídos, e o ruído decorado por uma é contradito pelas outras 49 na votação. O que sobrevive à média é o que estava presente na maioria das reamostras — ou seja, o sinal.
>
> O **sorteio de atributos** acrescenta um ganho a mais, e o mecanismo é sutil: sem ele, as 50 árvores tendem a escolher os mesmos primeiros atributos, porque a reamostra muda pouco o ranking dos melhores. Árvores parecidas erram parecido, e votar entre modelos que erram juntos não corrige nada. Forçar cada divisão a escolher entre poucos atributos sorteados **descorrelaciona** as árvores, e é a correlação entre elas, não a qualidade individual de cada uma, que limita o ganho de um comitê.
>
> É por isso que uma floresta aleatória usa **árvores deliberadamente piores** que a melhor árvore possível: individualmente cada uma acerta menos, e é justamente por isso que elas erram em lugares diferentes. O comitê é forte porque os erros não se somam.

Isso é um exemplo de uma técnica mais ampla chamada **aprendizado de comitê** (*ensemble learning*): combinar muitos modelos medíocres para produzir um bom. E vale dizer com precisão que tipo de modelo medíocre a floresta combina, porque é fácil errar isso. Ela combina árvores de **viés baixo e variância alta** — árvores que crescem até decorar o próprio treino, exatamente como os dois primeiros números da medição acima mostram —, e o que a média entre elas reduz é a **variância**. Reamostrar é uma imitação barata de "obter mais dados", que é a saída que a [seção 8.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/05-vies-e-variancia.html) indica para variância alta, e não para viés alto.

> **⚠️ Atenção — Aprendiz fraco: viés alto ou variância alta?**
>
> Uma frase que circula bastante diz que o aprendizado de comitê combina vários aprendizes fracos, "tipicamente modelos de **viés alto e variância baixa**". Para a floresta aleatória, isso está trocado — e vale desfazer a troca com cuidado, porque a descrição errada descreve com exatidão **outro** método.
>
> Viés alto com variância baixa é a definição de *underfitting* da [seção 8.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/05-vies-e-variancia.html): um modelo que erra sistematicamente e erra a mesma coisa em qualquer conjunto de treino. Uma árvore sem poda é o oposto exato disso, e este capítulo acabou de medir: 99,7% no treino contra 82,4% em dados novos é viés baixíssimo com variância altíssima. E se as árvores de uma floresta tivessem mesmo viés alto e variância baixa, tirar a média entre elas não adiantaria nada — elas errariam todas do mesmo jeito, e a média de erros idênticos é o mesmo erro. A floresta só funciona porque a premissa da frase é falsa.
>
> O método que de fato parte de modelos de viés alto é o **boosting**, mencionado no fim desta seção. Ele usa árvores propositalmente rasas — às vezes com uma pergunta só — e as encadeia, cada uma treinada para corrigir o resíduo das anteriores, reduzindo **viés**. Floresta e boosting atacam pontas opostas do mesmo compromisso, e a expressão *weak learner*, que é vocabulário de boosting, é provavelmente o que arrasta a frase para o lado errado.

### Então quando é que a floresta ganha?

O ganho da floresta sobre a árvore **sem poda** é real, e é ganho de variância: 82,4% para 92,8%, com as duas fontes de aleatoriedade contribuindo separadamente. Nada disso muda. O que muda é a moral da história, porque a árvore de duas perguntas ganhou de todas as florestas por sete pontos percentuais — e é um modelo que se desenha numa página.

A objeção óbvia é que escolhemos a profundidade 2 sabendo a regra verdadeira, o que ninguém pode fazer na vida real. Ela tem resposta, e a resposta é medível: separe uma parte do treino como conjunto de **validação**, escolha a profundidade por ela, e não olhe o teste — a receita da [seção 8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html).

In [ ]:
random.seed(1)

PROFUNDIDADES = [1, 2, 3, 4, 5, 8]
escolhidas: Counter = Counter()
no_teste = []

for _ in range(20):
    treino = gera(200)
    teste = gera(1000, ruido=0.0)
    ajuste, validacao = treino[:150], treino[150:]   # 150 para ajustar, 50 para escolher

    def nota(d: int) -> float:
        candidata = build_tree_podada(ajuste, ATRIBUTOS, 'alvo', d)
        return acuracia(lambda linha: classify(candidata, linha), validacao)

    melhor = max(PROFUNDIDADES, key=nota)
    escolhidas[melhor] += 1

    final = build_tree_podada(treino, ATRIBUTOS, 'alvo', melhor)
    no_teste.append(acuracia(lambda linha: classify(final, linha), teste))

print(f"profundidades escolhidas: {dict(sorted(escolhidas.items()))}")
print(f"acurácia no teste: média={sum(no_teste) / len(no_teste):.4f}  "
      f"pior={min(no_teste):.4f}")

Cinquenta linhas de validação bastam para encontrar a profundidade 2, nas 20 repetições, sem que ninguém tenha contado a regra ao procedimento. Em 19 delas a árvore resultante acerta o teste inteiro; numa, o ruído do treino faz o critério de entropia escolher um atributo errado na raiz e ela cai para 87,9%. A média fica em 99,4% — ainda muito acima de qualquer floresta.

> **🔷 Conceito**
>
> A conclusão honesta deste experimento não é a que se costuma contar.
>
> Quando a estrutura verdadeira é simples o bastante para caber numa árvore rasa, a resposta mais barata **não** é a floresta: é uma regra de parada. Ela custa uma linha de código, é encontrada por um conjunto de validação, e devolve um modelo que continua legível.
>
> A floresta ganha quando não se sabe de antemão que a estrutura é simples — que é o caso comum. Ela chegou a 92,8% aqui sem que ninguém lhe dissesse nada sobre o problema. E há um caso em que a vantagem da árvore podada desaparece por construção: se a regra verdadeira envolvesse seis atributos em vez de dois, nenhuma profundidade rasa a expressaria, e podar deixaria de ser resposta. Este experimento não mede esse caso, e vale registrar que não mede.

### O que a floresta cobra

Vale terminar o capítulo onde ele começou.

A [seção 14.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/01-o-que-e-uma-arvore-de-decisao.html) abriu dizendo que a árvore é o primeiro classificador aprendido a partir de dados, neste livro, que um humano lê — que a figura **é** o modelo, e que dá para discordar dela. As cem árvores acima, tomadas em conjunto, não são legíveis por ninguém. Não existe figura. A resposta para "por que este candidato foi classificado assim?" passou de um caminho de duas perguntas para "porque 78 de 100 árvores, cada uma treinada num subconjunto sorteado dos dados usando um subconjunto sorteado dos atributos, votaram assim".

> **❗ Importante**
>
> Esse é o compromisso: a floresta aleatória compra generalização com a única moeda que tornava a árvore especial.
>
> Só que ela nem sempre precisa pagar. A medição desta seção mostrou um conjunto em que o modelo mais legível é também o mais preciso — a árvore de duas perguntas —, e o preço da legibilidade ali é zero. Quando esse for o caso, a troca não existe, e supor que ela existe custa acurácia além de custar transparência. **Vale medir antes de assumir.**
>
> Quando a troca existe de fato, não há resposta universal sobre qual lado escolher. Se o modelo precisa ser auditado por um humano — decisões de crédito, triagem médica, admissão —, uma árvore rasa que se lê numa página pode valer mais do que alguns pontos de acurácia. Se ele só precisa acertar, a floresta ganha quase sempre.
>
> O que **não** vale é fingir que não houve troca. Existe uma indústria inteira de ferramentas de "explicabilidade" para modelos de comitê, e todas elas produzem explicações **aproximadas**, reconstruídas por fora do modelo. A explicação da árvore da [seção 14.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/04-criando-uma-arvore.html) não era uma aproximação: era o modelo.

E é aqui que o capítulo entrega o aluno ao próximo. A árvore comprou não linearidade **de graça**: ela nunca precisou de produto escalar, de coeficiente, de reescalonamento nem do gradiente descendente do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html), porque perguntar "qual é o valor?" já dobra a fronteira sozinho. O que ela pagou foi legibilidade, e pagou só quando virou floresta.

O [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) compra a mesma não linearidade de volta ao gradiente — a rede neural é uma pilha de funções deriváveis, e todo o maquinário que este capítulo dispensou volta inteiro, taxa de aprendizado inclusive. E o preço, dessa vez, vem cobrado adiantado: o modelo que sai de lá não é legível nem de longe, nem por aproximação, nem quando é um só.

> **💡 Dica — Na prática: `RandomForestClassifier`**
>
> Tudo desta seção, em quatro linhas:
>
> ```python
> from sklearn.ensemble import RandomForestClassifier
>
> modelo = RandomForestClassifier(
>     n_estimators=100,     # quantas árvores
>     criterion='entropy',  # o critério de divisão deste capítulo
>     max_features='sqrt',  # atributos sorteados por divisão (o padrão)
>     bootstrap=True,       # reamostrar os dados (o padrão)
>     oob_score=True,       # estimar o desempenho com os exemplos não sorteados
>     random_state=0,
> ).fit(X, y)
>
> modelo.oob_score_          # acurácia estimada sem separar conjunto de teste
> modelo.feature_importances_
> ```
>
> Os dois parâmetros do meio são exatamente as duas fontes de aleatoriedade desta seção, e o padrão de `max_features` merece uma conta feita à mão. `'sqrt'` **não** arredonda: a regra é `max(1, int(sqrt(p)))`, e com os oito atributos deste experimento $\sqrt{8} \approx 2{,}83$ é **truncado para 2**. A chamada acima reproduz, portanto, a linha `floresta (2 de 8)` da nossa tabela — 91,7% —, e não a melhor, `floresta (3 de 8)`, com 92,8%.
>
> Guarde o tamanho disso: o padrão da biblioteca ficou 1,2 ponto percentual abaixo do melhor valor que medimos, num problema pequeno e sem nada escondido. A "recomendação clássica" $\sqrt{p}$ para classificação é uma regra de bolso boa o bastante para começar, não um ótimo — e `max_features` é justamente um dos primeiros parâmetros a varrer quando o último ponto percentual importa.
>
> Duas coisas que a biblioteca dá e que não construímos:
>
> - **`predict_proba`** devolve a fração de árvores que votaram em cada classe, que é a contagem de votos da nossa tabela normalizada. É uma medida de concordância do comitê, não uma probabilidade calibrada — 78 de 100 árvores não quer dizer 78% de chance.
> - **`feature_importances_`** soma, para cada atributo, a redução de impureza que ele produziu em todas as divisões de todas as árvores. É útil, e tem um viés conhecido e importante: **favorece atributos com muitos valores distintos**, pelo mesmo motivo que a [seção 14.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/03-a-entropia-de-uma-particao.html) mostrou com a coluna de matrícula. A alternativa recomendada é a *permutation importance*, que mede a queda de desempenho ao embaralhar a coluna.
>
> Se você for além disto, o próximo passo natural não é uma floresta maior: é o **gradient boosting** (`HistGradientBoostingClassifier`, `XGBoost`, `LightGBM`), que constrói árvores em **sequência**, cada uma treinada para corrigir os erros das anteriores. A floresta reduz variância combinando modelos independentes; o boosting reduz viés encadeando modelos dependentes. É outra ideia, e hoje é ela que costuma ganhar em dados tabulares.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 17 de Grus (2019) faz três sugestões.

A primeira é o `scikit-learn`, que traz vários [modelos de árvore de decisão](https://scikit-learn.org/stable/modules/tree.html) e um [módulo `ensemble`](https://scikit-learn.org/stable/modules/ensemble.html) com o `RandomForestClassifier` e outros métodos de comitê. Os callouts de fechamento das seções [14.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/05-juntando-tudo.html) e [14.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html) mostram as duas interfaces — e apontam a diferença estrutural entre a árvore do `scikit-learn` (CART: divisões binárias sobre atributos numéricos) e a que este capítulo constrói (ID3: uma divisão por valor de um atributo categórico).

A segunda é o [XGBoost](https://xgboost.readthedocs.io/), biblioteca de *gradient boosted trees* — árvores construídas em sequência, cada uma corrigindo o erro das anteriores — que domina competições de aprendizado de máquina sobre dados tabulares. É um parente próximo da floresta aleatória da [seção 14.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html), com uma diferença essencial: a floresta constrói suas árvores em paralelo e independentes, o *boosting* as constrói em série e dependentes.

A terceira é a [Wikipédia](https://en.wikipedia.org/wiki/Decision_tree_learning), como ponto de partida para a variedade de algoritmos de árvore que este capítulo não cobre — C4.5, CART, CHAID — e para o problema de poda, que aqui aparece na sua forma mais simples — um limite de profundidade — na [seção 14.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html).

Para o tratamento estatístico das árvores e dos métodos de comitê, James et al. (2021) é o ponto de partida usual, e Hastie et al. (2009) dedica capítulos inteiros a *bagging*, florestas aleatórias e *boosting*.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed.. Springer. 2009.
- **James; Witten; Hastie; Tibshirani**. *An Introduction to Statistical Learning*. 2nd ed.. Springer. 2021.